# HmmPy — Probabilistic Decision Layer for OpenAI-Compatible Models

**Developer:** Pouriya Khalilian  
**Demo notebook:** Financial decision scenarios  
**Target version:** HmmPy 0.3.3

---

## What is HmmPy?

HmmPy is a lightweight Python library that turns OpenAI-compatible language models into a typed probabilistic decision layer.

Instead of only returning free-form text, HmmPy can return structured decisions such as:

- `BooleanResult`
- `ChoiceResult`
- `ScoreResult`
- `AutoResult`
- `AbstainResult`

It also supports:

- OpenAI-compatible cloud providers
- Local OpenAI-compatible servers
- Stateful chat sessions
- Optional conversation history
- Multiple-sample probability aggregation
- Confidence thresholds
- Decision policies
- Abstention
- Escalation with additional samples
- Token usage tracking
- Cost estimation

> Important: HmmPy probabilities are model-derived uncertainty estimates. They are not statistically calibrated probabilities unless a separate calibration process is added.

## Demo objective

This notebook demonstrates HmmPy using financial examples where the expected answer is known in advance.

That makes the examples useful as a controlled sanity check:

1. Positive NPV → expected answer: `True`
2. Negative NPV → expected answer: `False`
3. Project selection → expected answer can be checked from the supplied numbers
4. Risk scoring → demonstrates `ScoreResult`
5. Auto decision type → demonstrates automatic schema inference
6. Strict policy → demonstrates abstention and escalation

## 1. Setup

Place `hmmpy.py` in the same directory as this notebook.

Install the only required runtime dependency:

```bash
pip install -U openai
```

In [ ]:
import hmmpy
from hmmpy import HmmSession, DecisionPolicy

print("HmmPy version:", hmmpy.__version__)
print("Loaded from:", hmmpy.__file__)

Expected version:

```text
0.3.3
```

If an older version is shown, replace the local `hmmpy.py` file and restart the Jupyter kernel.

## 2. Provider configuration

HmmPy only requires an **OpenAI-compatible** endpoint.

The provider can be:

- a hosted API gateway
- a direct model provider
- LM Studio
- vLLM
- Ollama through an OpenAI-compatible endpoint
- another compatible local/server implementation

For the demo, fill in your own API key.

In [ ]:
API_KEY = "YOUR_API_KEY"
BASE_URL = "https://api.gapgpt.app/v1"
MODEL = "gpt-5.4"

hmm = HmmSession(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    timeout=300,
    max_retries=2,
)

hmm.set_system_prompt(
    """
    You are a careful probabilistic decision engine.
    Analyze only the evidence provided.
    Be conservative with uncertainty.
    """
)

print(hmm.config())

## 3. Scenario A — Positive NPV

A project requires an initial investment of **$10 million**.

It generates a single cash inflow of **$13 million after one year**.

Required return = **10%**

The present value is:

\[
PV = \frac{13}{1.10} \approx 11.818
\]

Therefore:

\[
NPV = 11.818 - 10 \approx +1.818
\]

So the correct answer is:

**Positive NPV = True**

In [ ]:
positive_npv_scenario = """
A company is considering a project.

Project details:
- Initial investment: $10 million
- Cash inflow after one year: $13 million
- No additional cash flows
- Required return: 10%
- Ignore taxes and financing effects

Question:
Does this project have a positive net present value?
"""

positive_result = hmm.boolean(
    positive_npv_scenario,
    samples=5,
    min_probability=0.90,
    min_confidence=0.80,
    use_history=False,
)

print("Result object:")
print(positive_result)
print()
print("Value:", positive_result.value)
print("Probability:", positive_result.probability)
print("Confidence:", positive_result.confidence)
print("Failed checks:", positive_result.failed_checks)
print("Usage:", positive_result.usage)

### Expected result

```text
Value: True
Failed checks: {}
```

The probability and confidence may vary by model, but the decision should be `True`.

## 4. Scenario B — Negative NPV

Now change the future cash inflow to **$9 million**.

\[
PV = \frac{9}{1.10} \approx 8.182
\]

\[
NPV = 8.182 - 10 \approx -1.818
\]

So the correct answer is:

**Positive NPV = False**

In [ ]:
negative_npv_scenario = """
A company is considering a project.

Project details:
- Initial investment: $10 million
- Cash inflow after one year: $9 million
- No additional cash flows
- Required return: 10%
- Ignore taxes and financing effects

Question:
Does this project have a positive net present value?
"""

negative_result = hmm.boolean(
    negative_npv_scenario,
    samples=5,
    min_probability=0.90,
    min_confidence=0.80,
    use_history=False,
)

print("Result object:")
print(negative_result)
print()
print("Value:", negative_result.value)
print("Probability:", negative_result.probability)
print("Confidence:", negative_result.confidence)
print("Failed checks:", negative_result.failed_checks)
print("Usage:", negative_result.usage)

### Expected result

```text
Value: False
Failed checks: {}
```

This pair of tests is useful because both cases are mathematically determined by the supplied inputs.

## 5. Scenario C — Choice decision

Suppose management must choose between three mutually exclusive one-year projects.

All require an initial investment of **$10 million** and use a required return of **10%**.

- Project A → $11 million after one year
- Project B → $13 million after one year
- Project C → $10.5 million after one year

Project B has the highest present value and therefore the highest NPV.

HmmPy can express the decision as a categorical `ChoiceResult`.

In [ ]:
choice_scenario = """
A company must choose one of three mutually exclusive projects.

Each project requires an initial investment of $10 million.
Required return is 10%.
There are no other cash flows.

Project A:
- Cash inflow after one year: $11 million

Project B:
- Cash inflow after one year: $13 million

Project C:
- Cash inflow after one year: $10.5 million

Which project has the highest NPV?
"""

choice_result = hmm.choice(
    choice_scenario,
    choices=["Project A", "Project B", "Project C"],
    samples=5,
    min_probability=0.80,
    min_confidence=0.60,
    use_history=False,
)

print("Selected:", choice_result.value)
print("Probability:", choice_result.probability)
print("Confidence:", choice_result.confidence)
print("Distribution:")

for name, p in choice_result.probabilities.items():
    print(f"  {name}: {p:.4f}")

print("Failed checks:", choice_result.failed_checks)

### Expected result

```text
Selected: Project B
```

The full probability distribution is useful when the top alternatives are close.

## 6. Scenario D — Score decision

HmmPy can also return an ordinal or numerical score distribution.

In this example, the model rates project attractiveness on a 1–5 scale based only on the supplied financial facts.

This is not a deterministic accounting formula like NPV; it demonstrates the `ScoreResult` interface.

In [ ]:
score_scenario = """
Evaluate the financial attractiveness of the following project.

- Initial investment: $10 million
- Cash inflow after one year: $13 million
- Required return: 10%
- No other cash flows
- No uncertainty in the supplied cash flow

Rate the project's financial attractiveness from 1 to 5,
where 1 = very unattractive and 5 = very attractive.
"""

score_result = hmm.score(
    score_scenario,
    scale=[1, 2, 3, 4, 5],
    samples=5,
    use_history=False,
)

print("Expected score:", score_result.value)
print("Confidence:", score_result.confidence)
print("Distribution:")

for score, p in score_result.distribution.items():
    print(f"  {score}: {p:.4f}")

print("Failed checks:", score_result.failed_checks)

## 7. Auto schema inference

`auto()` first decides what type of decision is appropriate, then executes that decision.

For a yes/no financial question, it should infer a boolean schema automatically.

In [ ]:
auto_result = hmm.auto(
    """
    A project requires $10 million today and returns $13 million
    exactly one year later. Required return is 10%.

    Does the project have a positive NPV?
    """,
    samples=3,
    min_probability=0.85,
    min_confidence=0.70,
    use_history=False,
)

print("Detected kind:", auto_result.kind)
print("Value:", auto_result.value)
print("Confidence:", auto_result.confidence)
print("Inferred spec:", auto_result.inferred_spec)
print("Failed checks:", auto_result.failed_checks)

### Expected result

```text
Detected kind: boolean
Value: True
```

## 8. Strict decision policy and abstention

HmmPy can refuse to accept a decision when the probability or confidence does not meet policy thresholds.

This is useful for ambiguous or weak-evidence situations.

The next scenario intentionally contains insufficient evidence.

In [ ]:
strict_policy = DecisionPolicy(
    min_probability=0.93,
    min_confidence=0.90,
    on_uncertain="abstain",
    escalate=True,
    escalation_samples=(5, 9),
)

uncertain_scenario = """
A startup has shown an early prototype to two potential users.
Both users gave positive informal feedback.

There is:
- no sales data
- no retention data
- no market study
- no validated pricing data
- no statistically meaningful customer sample

Question:
Is there enough evidence to confidently conclude that the startup will succeed?
"""

before_requests = hmm.usage().requests

strict_result = hmm.boolean(
    uncertain_scenario,
    samples=3,
    policy=strict_policy,
    use_history=False,
)

after_requests = hmm.usage().requests

print(strict_result)
print()
print("Requests used:", after_requests - before_requests)
print("Usage:", strict_result.usage)

if hmm.is_abstained(strict_result):
    print("\nHmmPy abstained.")
    print("Reason:", strict_result.reason)
    print("Failed checks:", strict_result.failed_checks)
    print("Candidate decision:", strict_result.candidate.value)
    print("Candidate probability:", strict_result.probability)
    print("Candidate confidence:", strict_result.confidence)
else:
    print("\nDecision accepted.")
    print("Value:", strict_result.value)
    print("Failed checks:", strict_result.failed_checks)

### Why abstention matters

A decision engine should not be forced to make a confident claim when the evidence is weak.

HmmPy can:

1. evaluate the initial samples
2. detect that thresholds are not satisfied
3. escalate to more samples
4. still abstain if uncertainty remains too high

This makes uncertainty a first-class part of the API.

## 9. Conversation history vs isolated decisions

By default in HmmPy 0.3.3, decisions do **not** use prior chat history.

This reduces:

- token usage
- accidental context contamination
- unnecessary cost

For a conversational decision, history can be enabled explicitly.

In [ ]:
hmm.chat(
    """
    The internal project codename is Project Aurora.
    Its maximum approved budget is $12 million.
    """
)

with_history = hmm.boolean(
    "Based on our previous conversation, is the maximum approved budget $12 million?",
    samples=3,
    use_history=True,
)

without_history = hmm.boolean(
    "Based on our previous conversation, is the maximum approved budget $12 million?",
    samples=3,
    use_history=False,
)

print("WITH HISTORY")
print("Value:", with_history.value)
print("Input tokens:", with_history.usage.prompt_tokens)
print()

print("WITHOUT HISTORY")
print("Value:", without_history.value)
print("Input tokens:", without_history.usage.prompt_tokens)

## 10. Usage and cost tracking

Token usage is tracked automatically when the provider returns OpenAI-compatible usage information.

In [ ]:
print("Last usage:")
print(hmm.last_usage())

print("\nWhole session:")
print(hmm.usage())

print("\nTotal tokens:")
print(hmm.tokens)

print("\nSession stats:")
print(hmm.stats())

### Optional cost estimation

Pricing is not standardized by the OpenAI-compatible protocol, so HmmPy lets the developer configure provider pricing manually.

Example:

```python
hmm.set_pricing(
    input_per_million=0.50,
    output_per_million=2.00,
)

print(hmm.cost())
```

This allows the same HmmPy interface to work with arbitrary providers.

## 11. Summary

This notebook demonstrates the current HmmPy architecture:

```text
OpenAI-compatible provider
          ↓
      HmmSession
          ↓
  typed decision primitive
          ↓
  multi-sample inference
          ↓
 probability distribution
          ↓
 confidence + policy checks
          ↓
 accept / escalate / abstain
          ↓
 typed Python result
```

### Current capabilities demonstrated

- OpenAI-compatible provider abstraction
- Boolean decisions
- Choice decisions
- Score decisions
- Automatic decision-type inference
- Probability aggregation
- Confidence estimation
- Decision policies
- Thresholds
- Abstention
- Sample escalation
- Conversation isolation
- Token accounting
- Optional cost estimation

### Next planned direction

A natural next stage is model routing:

```text
cheap model
    ↓ uncertain
balanced model
    ↓ uncertain
strong model
    ↓
decision / abstain
```

This would allow HmmPy to allocate model capability and cost based on the difficulty and importance of each decision.

---

## Disclaimer

The financial examples in this notebook are controlled software demonstrations with simplified inputs and known arithmetic outcomes.

They are not investment advice, portfolio recommendations, or real-world financial forecasts.

The purpose of the examples is to demonstrate the behavior of a probabilistic decision API.